In [76]:
import pandas as pd
import xml.etree.ElementTree as ET
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

## Data

### PPI

In [77]:
protein_interaction = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.v12.0.txt', sep= ' ')
protein_interaction_full = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.full.v12.0.txt', sep= ' ')
protein_interaction_detailed = pd.read_csv('Data/Protein-protein interaction data/9606.protein.links.detailed.v12.0.txt', sep= ' ')
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

In [78]:
### convert proteins to their true names
protein_info = pd.read_csv('Data/Protein-protein interaction data/9606.protein.info.v12.0.txt', on_bad_lines='skip', sep='\t')
protein_aliases= pd.read_csv('Data/Protein-protein interaction data/9606.protein.aliases.v12.0.txt', on_bad_lines='skip', sep='\t')

# Method 1: Using the to_dict() method with 'index' as orient
protein_info_translate_name_dict = protein_info.set_index('#string_protein_id')['preferred_name'].to_dict()
protein_alias_translate_name_dict = protein_aliases.set_index('#string_protein_id')['alias'].to_dict()
#print(protein_info_translate_name_dict)

### Protein1
protein1_name = []
for prot_id in tqdm(protein_interaction['protein1']):
    if prot_id in protein_info_translate_name_dict:
        protein1_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein1_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein1_name.append('')

### Protein 2
protein2_name = []
for prot_id in tqdm(protein_interaction['protein2']):
    if prot_id in protein_info_translate_name_dict:
        protein2_name.append(protein_info_translate_name_dict[prot_id])
    elif prot_id in protein_alias_translate_name_dict:
        protein2_name.append(protein_alias_translate_name_dict[prot_id])
    else:
        protein2_name.append('')

protein_interaction['Translated_protein_1'] = protein1_name
protein_interaction['Translated_protein_2'] = protein2_name

# Create a set of all (protein1, protein2) pairs
ppi_pairs = set(zip(protein_interaction['Translated_protein_1'], protein_interaction['Translated_protein_2']))
# Check for missing reverse pairs
missing_reverse = []
for a, b in ppi_pairs:
    if (b, a) not in ppi_pairs:
        missing_reverse.append((a, b))

print(f"Number of pairs missing their reverse: {len(missing_reverse)}")
if missing_reverse:
    print("Examples:", missing_reverse[:10])
else:
    print("All pairs have their reverse present.")

100%|██████████| 13715404/13715404 [00:02<00:00, 5402032.02it/s]


Number of pairs missing their reverse: 0
All pairs have their reverse present.


In [79]:
protein_interaction

,protein1,protein2,combined_score,Translated_protein_1,Translated_protein_2
0,9606.ENSP00000000233,9606.ENSP00000356607,173,ARF5,RALGPS2
1,9606.ENSP00000000233,9606.ENSP00000427567,154,ARF5,FHDC1
2,9606.ENSP00000000233,9606.ENSP00000253413,151,ARF5,ATP6V1E1
3,9606.ENSP00000000233,9606.ENSP00000493357,471,ARF5,CYTH2
4,9606.ENSP00000000233,9606.ENSP00000324127,201,ARF5,PSD3
...,...,...,...,...,...
13715399,9606.ENSP00000501317,9606.ENSP00000475489,195,RFX7,MPHOSPH9
13715400,9606.ENSP00000501317,9606.ENSP00000370447,158,RFX7,VCX
13715401,9606.ENSP00000501317,9606.ENSP00000312272,226,RFX7,YPEL2
13715402,9606.ENSP00000501317,9606.ENSP00000402092,169,RFX7,SAMD3


### DrugBank

In [80]:
# import xml.etree.ElementTree as ET

# # Load XML
# drugbank_xml = 'Data/DGIDB/drug_bank.xml'
# tree = ET.parse(drugbank_xml)
# root = tree.getroot()

# # Namespace
# ns = {'db': 'http://www.drugbank.ca'}

# Helper to clean tag names
def clean_tag(tag):
    return tag.split('}')[-1] if '}' in tag else tag

# Recursive function to print structure
def print_structure(elem, level=0):
    indent = '  ' * level
    print(f"{indent}- {clean_tag(elem.tag)}")
    for child in elem:
        print_structure(child, level + 1)

# # Get first drug
# first_drug = root.find('db:drug', ns)

# print("🌿 Structure of First Drug Entry:")
# print_structure(first_drug)
# print("\n🌳 Structure of First 3 Drug Entries:")
# drugs = root.findall('db:drug', ns)

# for i, drug in enumerate(drugs[:3]):
#     print(f"\n🔬 Drug {i+1}:")
#     print_structure(drug)


In [81]:
def structure_drug_bank_data(drug_bank_file = 'Data/DGIDB/drug_bank.xml'):
    """
    Function to structure the drug bank data from the XML file.
    :param drug_bank_file: Path to the drug bank XML file.
    :return: DataFrame containing structured drug bank data.
    """
    ### FYI the .find command only finds the first instance of a tag, 
    ### while .findall retrieves all instances of the specified tag within the current element.

    tree = ET.parse(drug_bank_file)
    root = tree.getroot()

    # DrugBank uses a specific namespace
    ns = {'db': 'http://www.drugbank.ca'}
    ### extract all drug elements
    drugs = root.findall('db:drug', ns)
    print(f"Found {len(drugs)} drugs in the DrugBank XML.")
    # Extract drug-gene interactions
    interactions = []
    # The interactions list will store dictionaries with 'drug' and 'gene' keys.
    for drug in root.findall('db:drug', ns): # root.findall('db:drug', ns): Finds all <drug> elements using the namespace.
        drug_name  = drug.find('db:name', ns).text  # drug.find('db:name', ns): Gets the drug's name.
        # print(drug_name)
        for target in drug.findall('db:targets/db:target', ns):  # drug.findall('db:targets/db:target', ns): Finds all <target> elements within <targets>.
            # print(target.tag)
            gene_description = target.find('db:name', ns)  # target.find('db:name', ns): Extracts the gene name for each target.
            poly = target.find('db:polypeptide', ns)  # target.find('db:polypeptide', ns): Extracts the polypeptide information.
            action = target.find('db:actions/db:action', ns) # target.find('db:actions/db:action', ns): Extracts the action of the drug on the target.
            if poly is not None:
                poly_name = poly.find('db:name', ns)
                gene_name = poly.find('db:gene-name', ns)
                specific_function = poly.find('db:specific-function', ns)
                interactions.append({
                    'drug': drug_name,
                    'polypeptide': poly_name.text if poly_name is not None else None,
                    'gene': gene_name.text if gene_name is not None else None,
                    'gene_description': gene_description.text if gene_description is not None else None,
                    'action': action.text if action is not None else None,
                    'specific_function': specific_function.text if specific_function is not None else None
                })
            ############# if polypeptide is not present, we still want to add the drug and gene information
            ############# this is because some drugs may not have a polypeptide associated with them
            ############# but we still want to capture the drug and gene information
            ############# this is common in the DrugBank database, where some drugs target genes directly
            ############# and do not have a polypeptide associated with them

            else:
                gene_name = None
                specific_function = None
                poly_name = None
                action = None
                gene_description = None
                resource = None
                identifier = None
  
                interactions.append({
                        'drug': drug_name,
                        'polypeptide': poly_name.text if poly_name is not None else None,
                        'gene': gene_name.text if gene_name is not None else None,
                        'gene_description': gene_description.text if gene_description is not None else None,
                        'action': action.text if action is not None else None,
                        'specific_function': specific_function.text if specific_function is not None else None
                    })
        
    # Convert to DataFrame
    # Converts the list of dictionaries into a pandas DataFrame, which is easier to analyze, filter, and export.
    df = pd.DataFrame(interactions)

    return df

In [82]:
Drug_bank = structure_drug_bank_data('Data/DGIDB/drug_bank.xml')

Found 17430 drugs in the DrugBank XML.


### Genetic results

In [83]:
### import data

### genes
hpv_positive_genes  = pd.read_csv('Results/CNV results/HPV positive CNV top genes.csv')
hpv_negative_genes = pd.read_csv('Results/CNV results/HPV negative CNV top genes.csv')

### drug candiates
hpv_positive_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Direct Drug Candidates Aggregated.csv')
hpv_positive_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Positive Top Indirect Drug Candidates Aggregated.csv')

hpv_negative_direct_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Direct Drug Candidates Aggregated.csv')
#### no direct drug candidates came from Deletions, only amplifications
hpv_negative_direct_drug_candidates['MUT_TYPE'] = 'AMPLIFICATION'
hpv_negative_indirect_drug_candidates = pd.read_csv('Results/CNV results/HPV Negative Top Indirect Drug Candidates Aggregated.csv')

### somatic mtuation
hpv_positive_som_genes = pd.read_csv('Results/SOM results/HPV positive top genes.csv')
hpv_positive_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_direct_drug_candidates_agg.csv')
hpv_positive_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_positive_som_top_indirect_drug_candidates_agg.csv')
hpv_positive_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

hpv_negative_som_genes = pd.read_csv('Results/SOM results/HPV negative top genes.csv')
hpv_negative_som_direct_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_direct_drug_candidates_agg.csv')
hpv_negative_som_direct_drug_candidates['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_indirect_drug_candidates = pd.read_csv('Results/SOM results/hpv_negative_som_top_indirect_drug_candidates_agg.csv')
hpv_negative_som_indirect_drug_candidates['MUT_TYPE'] = 'SOMATIC'

In [84]:
hpv_negative_som_direct_drug_candidates

,DRUG,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,GENE_TARGET,GENE_Cohort_Frequency,GENE_Normalized_Count,GENE_Normalized_Cohort_Frequency,GENE_SIGNIFICANT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,MUT_TYPE
0,Fludiazepam,3,17,17.647059,GABRA1,9,0.007237,0.005921,True,potentiator,GABA-A receptor activity,4.204050e-09,8.707242e-06,0.00001,0.004073,SOMATIC
1,Fludiazepam,3,17,17.647059,GABRB3,14,0.007531,0.007531,True,potentiator,GABA-A receptor activity,4.204050e-09,8.707242e-06,0.00001,0.004073,SOMATIC
2,Fludiazepam,3,17,17.647059,GABRG1,11,0.007885,0.007885,True,potentiator,GABA receptor binding,4.204050e-09,8.707242e-06,0.00001,0.004073,SOMATIC
3,Bryostatin 1,1,9,11.111111,CASP8,52,0.033670,0.029181,True,inhibitor,cysteine-type endopeptidase activity,2.350142e-05,6.796138e-03,0.00003,0.009066,SOMATIC
4,Meprobamate,1,9,11.111111,GABRA1,9,0.007237,0.005921,True,positive allosteric modulator,GABA-A receptor activity,1.022744e-04,1.916213e-02,0.00010,0.019932,SOMATIC
5,Secobarbital,1,11,9.090909,GABRA1,9,0.007237,0.005921,True,potentiator,GABA-A receptor activity,2.888105e-04,4.426144e-02,0.00026,0.045956,SOMATIC
6,Foreskin fibroblast (neonatal),1,11,9.090909,TGFBR2,19,0.011824,0.010698,True,agonist,activin binding,2.394028e-05,6.796138e-03,0.00004,0.011710,SOMATIC
7,Dasatinib,2,23,8.695652,EPHA5,22,0.007384,0.007063,True,inhibitor,ATP binding,5.499430e-08,5.724296e-05,0.00001,0.004073,SOMATIC
8,Dasatinib,2,23,8.695652,EPHA2,20,0.008197,0.006831,True,antagonist,ATP binding,5.499430e-08,5.724296e-05,0.00001,0.004073,SOMATIC
9,Talbutal,1,12,8.333333,GABRA1,9,0.007237,0.005921,True,positive allosteric modulator,GABA-A receptor activity,2.888105e-04,4.426144e-02,0.00029,0.049394,SOMATIC


#### Overlap

##### HPV+

In [85]:
### number of unique drugs of all hpv positive both direct and indirect
num_unique_pos_drugs = len(list(set(list(set(hpv_positive_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_pos_drugs

220

In [86]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print("\nHPV positive direct drug candidates:")
print(sorted(all_hpv_positive_direct_drugs))

Number of unique HPV positive direct drug candidates: 14

HPV positive direct drug candidates:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'copper', 'golotimod', 'nadh', 'tg-100801', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form']


In [87]:
# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive indirect drug candidates
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)

print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print("\nHPV positive indirect drug candidates:")
print(sorted(all_hpv_positive_indirect_drugs))

Number of unique HPV positive indirect drug candidates: 215

HPV positive indirect drug candidates:
['1-chloro-6-(4-hydroxyphenyl)-2-naphthol', '1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3,4-d]pyrimidin-4-ylamine', '2-[(2,4-dichlorobenzoyl)amino]-5-(pyrimidin-2-yloxy)benzoic acid', '2-chloro-5-nitro-n-phenylbenzamide', '2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imidazo[4,5-f]isoquinolin-7-one', '4-(4-methoxy-1h-pyrrolo[2,3-b]pyridin-3-yl)pyrimidin-2-amine', '4-(4-propoxy-1h-pyrrolo[2,3-b]pyridin-3-yl)pyrimidin-2-amine', '9,9,9-trifluoro-8-oxo-n-phenylnonanamide', '[4-({5-(aminocarbonyl)-4-[(3-methylphenyl)amino]pyrimidin-2-yl}amino)phenyl]acetic acid', '[5-hydroxy-2-(4-hydroxyphenyl)-1-benzofuran-7-yl]acetonitrile', 'abrocitinib', 'aceclidine', 'acetylsalicylic acid', 'aclidinium', 'afatinib', 'ag-24322', 'aleglitazar', 'alteplase', 'altiratinib', 'alvocidib', 'amuvatinib', 'an-9', 'anisotropine methylbromide', 'aripiprazole', 'aripiprazole lauroxil', 'arsenic trioxide', 'artenimol',

In [88]:
# Get the set of unique HPV positive direct drug candidates
hpv_positive_direct_drugs = set(hpv_positive_direct_drug_candidates['DRUG'].str.lower())
hpv_positive_som_direct_drugs = set(hpv_positive_som_direct_drug_candidates['DRUG'].str.lower())

# Get the set of unique HPV positive indirect drug candidates
hpv_positive_indirect_drugs = set(hpv_positive_indirect_drug_candidates['DRUG'].str.lower())
hpv_positive_som_indirect_drugs = set(hpv_positive_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV positive direct and indirect drug candidates
all_hpv_positive_direct_drugs = hpv_positive_direct_drugs.union(hpv_positive_som_direct_drugs)
all_hpv_positive_indirect_drugs = hpv_positive_indirect_drugs.union(hpv_positive_som_indirect_drugs)


# Find the overlap between direct and indirect drug candidates
overlapping_drugs = all_hpv_positive_direct_drugs.intersection(all_hpv_positive_indirect_drugs)

print(f"Number of unique HPV positive direct drug candidates: {len(all_hpv_positive_direct_drugs)}")
print(f"Number of unique HPV positive indirect drug candidates: {len(all_hpv_positive_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV positive direct drug candidates: 14
Number of unique HPV positive indirect drug candidates: 215
Number of overlapping drugs between direct and indirect: 9

Overlapping drugs:
['buparlisib', 'ch-5132799', 'cladribine', 'copanlisib', 'golotimod', 'nadh', 'tg-100801', 'wortmannin', 'xl765']


##### HPV-

In [89]:
### number of unique drugs of all hpv negative both direct and indirect
num_unique_neg_drugs = len(list(set(list(set(hpv_negative_direct_drug_candidates['DRUG'].str.lower()))
                           + list(set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())) 
                           + list(set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())))))
num_unique_neg_drugs

117

In [90]:
## HPV- Drug Candidates
### Number of unique direct drug candidates
# Get the set of unique HPV negative direct drug candidates
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative direct drug candidates
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)

print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print("\nHPV negative direct drug candidates:")
print(sorted(all_hpv_negative_direct_drugs))


Number of unique HPV negative direct drug candidates: 25

HPV negative direct drug candidates:
['acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'bryostatin 1', 'caffeine', 'copper', 'dasatinib', 'fludiazepam', 'foreskin fibroblast (neonatal)', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'heparin', 'meprobamate', 'metharbital', 'nadh', 'regorafenib', 'secobarbital', 'talbutal', 'wortmannin', 'xl765', 'zinc', 'zinc acetate', 'zinc chloride', 'zinc sulfate, unspecified form']


In [91]:
# Get the set of unique HPV negative indirect drug candidates
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine both sets to get all unique HPV negative indirect drug candidates
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)

print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print("\nHPV negative indirect drug candidates:")
print(sorted(all_hpv_negative_indirect_drugs))

Number of unique HPV negative indirect drug candidates: 111

HPV negative indirect drug candidates:
['3-isobutyl-1-methyl-7h-xanthine', 'aceclidine', 'acetylsalicylic acid', 'aclidinium', 'ag-24322', 'alsterpaullone', 'altiratinib', 'alvocidib', 'amuvatinib', 'aripiprazole lauroxil', 'arsenic trioxide', 'artenimol', 'bethanechol', 'biotin', 'bisindolylmaleimide i', 'bms-690514', 'brigatinib', 'bryostatin 1', 'caffeine', 'calcium citrate', 'calcium phosphate', 'calcium phosphate dihydrate', 'canertinib', 'carfilzomib', 'chlorprothixene', 'cholic acid', 'ci-1040', 'clozapine', 'conestat alfa', 'darifenacin', 'dasatinib', 'enzastaurin', 'erdafitinib', 'esflurbiprofen', 'famitinib', 'fesoterodine', 'fg-9041', 'fludiazepam', 'fn-1501', 'foreskin fibroblast (neonatal)', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'glycopyrronium', 'heparin', 'homatropine', 'homatropine methylbromide', 'human c1-esterase inhibitor', 'hymenialdisine', 'imatinib', 'lenvatinib'

In [92]:
# Get the set of unique HPV negative drug candidates (both direct and indirect)
hpv_negative_direct_drugs = set(hpv_negative_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_som_direct_drugs = set(hpv_negative_som_direct_drug_candidates['DRUG'].str.lower())
hpv_negative_indirect_drugs = set(hpv_negative_indirect_drug_candidates['DRUG'].str.lower())
hpv_negative_som_indirect_drugs = set(hpv_negative_som_indirect_drug_candidates['DRUG'].str.lower())

# Combine all HPV negative drug sets
all_hpv_negative_direct_drugs = hpv_negative_direct_drugs.union(hpv_negative_som_direct_drugs)
all_hpv_negative_direct_drugs = set(drug.lower() for drug in all_hpv_negative_direct_drugs)
all_hpv_negative_indirect_drugs = hpv_negative_indirect_drugs.union(hpv_negative_som_indirect_drugs)
all_hpv_negative_indirect_drugs = set(drug.lower() for drug in all_hpv_negative_indirect_drugs)

# Find the overlap between direct and indirect drug candidates for HPV negative
overlapping_drugs = all_hpv_negative_direct_drugs.intersection(all_hpv_negative_indirect_drugs)


print(f"Number of unique HPV negative direct drug candidates: {len(all_hpv_negative_direct_drugs)}")
print(f"Number of unique HPV negative indirect drug candidates: {len(all_hpv_negative_indirect_drugs)}")
print(f"Number of overlapping drugs between direct and indirect: {len(overlapping_drugs)}")
print("\nOverlapping drugs:")
print(sorted(overlapping_drugs))

Number of unique HPV negative direct drug candidates: 25
Number of unique HPV negative indirect drug candidates: 111
Number of overlapping drugs between direct and indirect: 19

Overlapping drugs:
['acetylsalicylic acid', 'biotin', 'bisindolylmaleimide i', 'bryostatin 1', 'caffeine', 'dasatinib', 'fludiazepam', 'foreskin fibroblast (neonatal)', 'foreskin keratinocyte (neonatal)', 'fostamatinib', 'gamma-aminobutyric acid', 'heparin', 'meprobamate', 'metharbital', 'nadh', 'regorafenib', 'secobarbital', 'talbutal', 'wortmannin']


#### Literature results

In [93]:
extracted_target_df= pd.read_csv('Validation pipeline/Results/cleaned_extracted_targets_all_pub_after_2000_GPU_2b_gemma.csv')
extracted_target_df_combined = pd.read_csv('Validation pipeline/Results/cleaned_extracted_combined_targets_all_pub_after_2000_GPU_2b_gemma.csv')

In [94]:
### accumulate all genes available in drugbank or ppi
Drug_bank_genes = list(Drug_bank['gene'].values)
ppi_genes = list(protein_interaction['Translated_protein_1'].values)
all_ppi_drugbank = list(set(Drug_bank_genes + ppi_genes))

## HPV+

#### Genes

In [95]:
hpv_positive_som_genes

,Gene,Count,Cohort_Frequency,Normalized_Count,Normalized_Cohort_Frequency,P_Value,Adjusted_P_Value,Significant,Empirical_P_Value,Adjusted_Empirical_P_Value,frequency_percentage,mutation_score
0,PIK3CA,18,17,0.005473,0.005169,4.738394e-19,2.079207e-15,True,0.0001,0.036563,23.611111,0.129219
1,ZNF750,11,8,0.005071,0.003688,7.324983e-12,1.607101e-08,True,0.0001,0.036563,11.111111,0.056350
2,CYLD,8,8,0.002677,0.002677,6.578872e-07,9.622697e-04,True,0.0001,0.036563,11.111111,0.029749
3,EP300,9,9,0.001243,0.001243,6.021135e-05,4.723959e-02,True,0.0001,0.036563,12.500000,0.015534
4,CCDC191,6,6,0.002083,0.002083,6.589348e-05,4.723959e-02,True,0.0001,0.036563,8.333333,0.017355
5,LRRC37B,6,5,0.001994,0.001662,8.343138e-05,4.723959e-02,True,0.0001,0.036563,6.944444,0.013847


In [96]:
### combine hpv positive somatic genes and cnv genes
hpv_positive_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_positive_som_genes['gene_name'] = hpv_positive_som_genes['Gene']
hpv_positive_som_genes['q_value']= hpv_positive_som_genes['Adjusted_P_Value']
hpv_positive_som_genes['empirical_q_value'] = hpv_positive_som_genes['Adjusted_Empirical_P_Value']
hpv_positive_combined_genes = pd.concat([hpv_positive_genes, hpv_positive_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_positive_combined_genes = hpv_positive_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()

hpv_positive_combined_genes


,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACE2,DELETION,8.234936784924659e-31,0.0203366143594484
1,ACTL6A,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
2,ADIPOQ,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
3,AHSG,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333
4,ANOS1,DELETION,8.234936784924659e-31,0.0203366143594484
5,BMX,DELETION,8.234936784924659e-31,0.0203366143594484
6,CCDC191,SOMATIC,0.0472395868080887,0.0365630103656301
7,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116600765426333
8,CNKSR2,DELETION,8.234936784924659e-31,0.0203366143594484
9,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301


In [97]:
len(set(hpv_positive_combined_genes['gene_name']))

51

In [98]:
### merge genes with number of articles, pubmed id from literature data
hpv_positive_genes_with_lit = pd.merge(hpv_positive_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_positive_genes_with_lit.drop(columns =['INDEX'], inplace = True)

In [99]:
hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

,gene_name,MUT_TYPE,q_value,empirical_q_value,GENE,PMID,NUMBER_OF_ARTICLES
7,CLDN1,AMPLIFICATION,6.5911053664735535e-53,0.0116600765426333,CLDN1,"15170668, 17091452",2.0
9,CYLD,SOMATIC,0.0009622697492491,0.0365630103656301,CYLD,"16900776, 18497946",2.0
31,PIK3CA,"AMPLIFICATION, SOMATIC","4.9013845356499375e-54, 2.079207200788839e-15","0.0116600765426333, 0.0365630103656301",PIK3CA,"11358835, 11836556, 11959846, 14581353, 155436...",17.0
36,RFC4,AMPLIFICATION,4.9013845356499375e-54,0.0116600765426333,RFC4,16467079,1.0
41,SOX2,AMPLIFICATION,1.2920437544143294e-55,0.0116600765426333,SOX2,15942670,1.0
43,TLR7,DELETION,8.234936784924659e-31,0.0203366143594484,TLR7,17201162,1.0


In [100]:
hpv_positive_genes_with_lit = hpv_positive_genes_with_lit[hpv_positive_genes_with_lit['NUMBER_OF_ARTICLES']>0]

In [101]:
hpv_positive_genes_with_lit.to_csv('Results/HPV positive gene results.csv')

In [102]:
### export to final results output
hpv_positive_genes_with_lit.to_csv('Results/Final Results/HPV Positive validated genes.csv')

#### Direct

In [103]:
### merge all hpv postive direct drug candidates
hpv_positive_final_direct = pd.concat([hpv_positive_direct_drug_candidates, hpv_positive_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value	drug_hypergeom_fdr	drug_empirical_p_value	drug_empirical_fdr	MUT_TYPE	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	GENE_SIGNIFICANT
hpv_positive_final_direct = hpv_positive_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x),
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [104]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00054,0.040148
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00054,0.040148
2,Cladribine,POLA1,DELETION,1,12,8.333333,inhibitor,chromatin binding,9.203648e-05,2.973096e-02,0.00015,0.041329
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.000000,inhibitor,1-phosphatidylinositol-3-kinase activity,5.168234e-04,3.433760e-02,0.00054,0.040148
4,Copper,"AHSG, KNG1","AMPLIFICATION, AMPLIFICATION",2,146,1.369863,None,cysteine-type endopeptidase inhibitor activity,8.218045e-15,2.566222e-11,0.00001,0.002532
5,Golotimod,TLR7,DELETION,1,5,20.000000,None,double-stranded RNA binding,1.605536e-05,6.539420e-03,0.00001,0.004930
6,NADH,NDUFB5,AMPLIFICATION,1,144,0.694444,None,NADH dehydrogenase (ubiquinone) activity,6.205177e-08,2.325204e-05,0.00001,0.002532
7,TG-100801,VEGFD,DELETION,1,8,12.500000,inhibitor,chemoattractant activity,4.038151e-05,1.576225e-02,0.00006,0.022483
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.555431e-05,5.766930e-03,0.00008,0.009861
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.000000,None,1-phosphatidylinositol-3-kinase activity,7.766063e-05,8.661009e-03,0.00008,0.012123


In [105]:
extracted_target_df_combined

,GENE,PMID,INDEX,NUMBER_OF_ARTICLES
0,000-2,"11302242, 11302242","2610, 2610",1
1,10,"12608845, 12768769, 14967420, 15193028, 180565...","15288, 17016, 27005, 29481, 57658, 61105",6
2,106PRE,18186293,58737,1
3,106R,18186293,58737,1
4,106RECR,18186293,58737,1
...,...,...,...,...
6072,ZP-V3,12464647,13458,1
6073,ZP-V4,12464647,13458,1
6074,ZYGOMA,15883929,36459,1
6075,ZYGOMATIC,"12775236, 17522494","17081, 53057",2


In [106]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES

hpv_positive_final_direct['PMID'] = ''
hpv_positive_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    hpv_positive_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(list(set(literature_gene_targets)))
    hpv_positive_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

In [107]:
hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00001,0.004930,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000046,0.005767,0.00008,0.009861,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00008,0.012123,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


In [108]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_positive_final_direct = hpv_positive_final_direct[hpv_positive_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_positive_final_direct.to_csv('Results/HPV Positive direct results.csv')

#### Indirect

In [109]:
### merge all hpv positive indirect drug candidates
hpv_positive_final_indirect = pd.concat([hpv_positive_indirect_drug_candidates, hpv_positive_som_indirect_drug_candidates], ignore_index=True)
hpv_positive_final_indirect['ACTION'] = hpv_positive_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['SPECIFIC_FUNCTION'] = hpv_positive_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_hypergeom_fdr'] = hpv_positive_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['drug_empirical_fdr'] = hpv_positive_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_positive_final_indirect['MUT_TYPE'] = hpv_positive_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### aggregate/group by drug name
### columns: DRUG	CONNECTED_TO (risk gene)	
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank	
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene	
# GENE_TARGET	GENE_Cohort_Frequency	
# GENE_Normalized_Count	GENE_Normalized_Cohort_Frequency	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr

hpv_positive_final_indirect = hpv_positive_final_indirect.groupby(['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()
hpv_positive_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(25)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
145,phthalic acid,"PGR, RXRA, ESR2, ESR1, PPARA, PPARG, RXRG",EP300,7,10,70.000000,UNKNOWN,"ATPase binding, DNA binding domain binding, DN...",6.082585e-05,0.002129,SOMATIC
149,pracinostat,"HDAC1, HDAC6, HDAC3, HDAC2",EP300,4,4,100.000000,UNKNOWN,"core promoter sequence-specific DNA binding, a...",1.487971e-03,0.002129,SOMATIC
7,"9,9,9-trifluoro-8-oxo-n-phenylnonanamide","HDAC10, HDAC4, HDAC1, HDAC6, HDAC2, HDAC8",EP300,6,7,85.714286,inhibitor,"acetylputrescine deacetylase activity, DNA-bin...",6.900589e-05,0.002129,SOMATIC
208,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.534269e-04,0.002129,SOMATIC
36,bosutinib,"HCK, FGR, MAP2K1, ABL1, LYN, SRC","PIK3CA, EP300",6,11,54.545455,"inhibitor, inhibitor","ATP binding, actin filament binding, ATP binding",1.552648e-03,0.002129,"SOMATIC, SOMATIC"
210,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,1.169028e-05,0.002129,SOMATIC
25,arsenic trioxide,"IKBKB, JUN, HDAC1, CDKN1A, MAPK1, CCND1, PML, ...","CYLD, EP300",8,10,80.000000,"inducer, inducer, antagonist","ATP binding, cAMP response element binding, co...",1.709173e-06,0.002129,"SOMATIC, SOMATIC"
24,aripiprazole lauroxil,"HTR1D, HTR1A, HTR7, DRD4, HTR1B, HTR1E, DRD2, ...",GNB4,13,25,52.000000,partial agonist,"G protein-coupled serotonin receptor activity,...",3.118399e-03,0.002532,AMPLIFICATION
26,artenimol,"GAPDH,NPM1,ANXA2,CCT3, HSPA8,RPS13, RPS6, EEF1...","SOX2,PIK3CA,AHSG,DNAJB11,EIF4A2,FXR1,HRG,RPL39...",36,104,34.615385,ligand,aspartic-type endopeptidase inhibitor activity,6.753139e-05,0.002532,"AMPLIFICATION,DELETION"
173,somatostatin,"SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1,OPRM1","SST,KNG1,GNB4",7,7,100.000000,agonist,"neuropeptide binding, G protein-coupled recept...",4.544467e-04,0.002532,AMPLIFICATION


In [110]:
hpv_positive_final_indirect[hpv_positive_final_indirect['DRUG']=='xl765']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
208,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.0,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",0.000153,0.002129,SOMATIC


In [111]:
hpv_positive_final_indirect[hpv_positive_final_indirect['ACTION']!= 'UNKNOWN'].sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
7,"9,9,9-trifluoro-8-oxo-n-phenylnonanamide","HDAC10, HDAC4, HDAC1, HDAC6, HDAC2, HDAC8",EP300,6,7,85.714286,inhibitor,"acetylputrescine deacetylase activity, DNA-bin...",6.900589e-05,0.002129,SOMATIC
25,arsenic trioxide,"IKBKB, JUN, HDAC1, CDKN1A, MAPK1, CCND1, PML, ...","CYLD, EP300",8,10,80.000000,"inducer, inducer, antagonist","ATP binding, cAMP response element binding, co...",1.709173e-06,0.002129,"SOMATIC, SOMATIC"
36,bosutinib,"HCK, FGR, MAP2K1, ABL1, LYN, SRC","PIK3CA, EP300",6,11,54.545455,"inhibitor, inhibitor","ATP binding, actin filament binding, ATP binding",1.552648e-03,0.002129,"SOMATIC, SOMATIC"
37,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC"
69,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002532,"AMPLIFICATION,DELETION, SOMATIC"
102,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",3.033567e-06,0.002532,"AMPLIFICATION,DELETION, SOMATIC"
108,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA,ANOS1,VEGFD, PIK3CA",8,8,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",8.743881e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC"
122,nintedanib,"FGFR1, FGFR3, SRC, KDR, PDGFRB, LCK, FLT1, PDG...","PIK3CA,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",12,12,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, ATP binding, ATP binding",2.018575e-07,0.002532,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC"
136,pd-166326,"FGFR1, SRC, PDGFRB, LCK, EGFR, KIT, ABL1, PDGF...","PIK3CA,GNB4,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",9,9,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",1.590798e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC"
173,somatostatin,"SSTR4, SSTR3, SSTR5, SSTR1, SSTR2,OPRD1,OPRM1","SST,KNG1,GNB4",7,7,100.000000,agonist,"neuropeptide binding, G protein-coupled recept...",4.544467e-04,0.002532,AMPLIFICATION


In [112]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_indirect_genes = extract_unique_gene_targets(hpv_positive_indirect_drug_candidates, 'GENE_TARGET')
print(hpv_pos_indirect_genes)

263


In [113]:
#### add columns to hpv_positive_final_direct for PMIds and NUMBER_OF_ARTICLES from extracted_target_df_combined
### ADD COLUMNS: PMIDs, NUMBER_OF_ARTICLES, gene
### combine based on gene target, and if any of the genes in GENE_TARGET are in extracted_target_df_combined, then add the PMIDs and NUMBER_OF_ARTICLES
hpv_positive_final_indirect['PMID'] = ''
hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

### validate risk genes
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect['RISK_GENE_PMID'] = ''
hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_positive_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_positive_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        #print(gene)
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_positive_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_positive_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### ensure that only drugs with NUMBER_OF_ARTICLES > 0 for both drug targets and risk genes are saved, so that they have literature support
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_positive_final_indirect = hpv_positive_final_indirect[hpv_positive_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

In [114]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)


186


In [115]:
hpv_positive_final_indirect.to_csv('Results/HPV Positive indirect results.csv', index=False)

In [116]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_pos_final_indirect_genes = extract_unique_gene_targets(hpv_positive_final_indirect, 'GENE_TARGET')
print(hpv_pos_final_indirect_genes)

186


In [117]:
hpv_positive_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
1,1-tert-butyl-3-(4-chloro-phenyl)-1h-pyrazolo[3...,"LCK, LYN, SRC","PIK3CA, EP300",3,3,100.000000,"UNKNOWN, UNKNOWN","ATP binding, ATP binding",0.012831,0.015613,"SOMATIC, SOMATIC","17252232, 17961551",2,"LYN, SRC","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
4,"2-tert-butyl-9-fluoro-1,6-dihydrobenzo[h]imida...","JAK2, TYK2, JAK3, JAK1",PIK3CA,4,5,80.000000,inhibitor,"acetylcholine receptor binding, ATP binding",0.005767,0.006788,SOMATIC,"15947106, 18204781",2,"JAK2, JAK3","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
10,abrocitinib,"JAK2, TYK2, JAK3, JAK1, JAK2, TYK2, JAK3, JAK1","PIK3CA, PIK3CA",4,4,100.000000,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",0.034338,0.040148,"AMPLIFICATION, SOMATIC","15947106, 18204781",4,"JAK2, JAK3","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
12,acetylsalicylic acid,"IKBKB, TP53, PCNA, MYC, NFKBIA, MAPK1, CCND1","CYLD, EP300",7,19,36.842105,"UNKNOWN, inducer, downregulator, inhibitor","ATP binding, 14-3-3 protein binding, chromatin...",0.008068,0.008862,"SOMATIC, SOMATIC","12854173, 15566678, 11854069, 15039910, 179089...",111,"NFKBIA, MYC, TP53, CCND1, PCNA","16900776, 18497946",2,CYLD
14,afatinib,"ERBB4, EGFR, ERBB2",PIK3CA,3,3,100.000000,inhibitor,"ATP binding, actin filament binding",0.012831,0.016756,SOMATIC,"17208308, 12833132, 18307254, 18202792, 168907...",325,"EGFR, ERBB2, ERBB4","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",0.000153,0.002129,SOMATIC,"15833854, 18316583, 17047074, 15379322, 169274...",8,MTOR,"18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
209,xl820,"KDR, PDGFRB, KIT, PDGFRA",PIK3CA,4,4,100.000000,UNKNOWN,ATP binding,0.001488,0.003123,SOMATIC,"17253141, 17935283, 16757203, 17513510, 144996...",9,"KIT, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
210,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,0.000012,0.002129,SOMATIC,"16209369, 14962731, 16350727, 10999773, 119531...",55,"RET, FGFR1, FGFR3","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
211,zanubrutinib,"JAK2, FGR, BTK, ERBB4, ITK, LCK, EGFR, JAK3, T...","PIK3CA, PIK3CA",10,15,66.666667,"inhibitor, inhibitor","acetylcholine receptor binding, ATP binding, a...",0.001983,0.005204,"AMPLIFICATION, SOMATIC","17208308, 12833132, 18307254, 18202792, 168907...",654,"EGFR, JAK2, ERBB2, JAK3, ERBB4","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA


In [118]:
hpv_positive_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
210,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,1.169028e-05,0.002129,SOMATIC,"16209369, 14962731, 16350727, 10999773, 119531...",55,"RET, FGFR1, FGFR3","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
25,arsenic trioxide,"IKBKB, JUN, HDAC1, CDKN1A, MAPK1, CCND1, PML, ...","CYLD, EP300",8,10,80.000000,"inducer, inducer, antagonist","ATP binding, cAMP response element binding, co...",1.709173e-06,0.002129,"SOMATIC, SOMATIC","16002527, 14697637, 18398822, 14603450, 167780...",34,"CCND1, PML, CDKN1A, HDAC1, AKT1","16900776, 18497946",2,CYLD
208,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,4,5,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.534269e-04,0.002129,SOMATIC,"15833854, 18316583, 17047074, 15379322, 169274...",8,MTOR,"18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
36,bosutinib,"HCK, FGR, MAP2K1, ABL1, LYN, SRC","PIK3CA, EP300",6,11,54.545455,"inhibitor, inhibitor","ATP binding, actin filament binding, ATP binding",1.552648e-03,0.002129,"SOMATIC, SOMATIC","17252232, 16224162, 17961551",3,"LYN, SRC, HCK","18447972, 14581353, 11358835, 15586224, 15...",17,PIK3CA
37,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",9,9,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC","17075124, 17208308, 12833132, 18307254, 182027...",725,"EGFR, ERBB2, MET, IGF1R, ALK, ERBB4","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
69,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002532,"AMPLIFICATION,DELETION, SOMATIC","16209369, 14962731, 16350727, 10999773, 119531...",145,"KIT, FGFR1, FGFR3, RET, FGFR2, FGFR4, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
102,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,ANOS1,VEGFD, PIK3CA",10,10,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",3.033567e-06,0.002532,"AMPLIFICATION,DELETION, SOMATIC","16209369, 14962731, 16350727, 10999773, 119531...",149,"KIT, FGFR1, FGFR3, RET, FGFR2, FGFR4, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
108,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA,ANOS1,VEGFD, PIK3CA",8,8,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",8.743881e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC","16807070, 15165306, 17545628, 11562460, 120752...",31,"FGFR1, FGFR2, FGFR3, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
122,nintedanib,"FGFR1, FGFR3, SRC, KDR, PDGFRB, LCK, FLT1, PDG...","PIK3CA,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",12,12,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, ATP binding, ATP binding",2.018575e-07,0.002532,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC","17252232, 16807070, 15165306, 17545628, 115624...",35,"SRC, FGFR1, LYN, FGFR3, FGFR2, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA
136,pd-166326,"FGFR1, SRC, PDGFRB, LCK, EGFR, KIT, ABL1, PDGF...","PIK3CA,GNB4,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",9,9,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",1.590798e-05,0.002532,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC","17538163, 12422313, 12509799, 12112535, 170780...",629,"KIT, EGFR, SRC, FGFR1, PDGFRA","18447972, 14581353, 11358835, 15586224, 15...",34,PIK3CA


#### overall

In [119]:
hpv_positive_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Buparlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
1,CH-5132799,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
3,Copanlisib,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,4,25.0,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000517,0.034338,0.00054,0.040148,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
5,Golotimod,TLR7,DELETION,1,5,20.0,None,double-stranded RNA binding,0.000016,0.006539,0.00001,0.004930,17201162,1,TLR7
8,Wortmannin,PIK3CA,SOMATIC,1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000046,0.005767,0.00008,0.009861,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
9,XL765,"PIK3CA, PIK3CA","AMPLIFICATION, SOMATIC",1,5,20.0,None,1-phosphatidylinositol-3-kinase activity,0.000078,0.008661,0.00008,0.012123,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


In [120]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_positive_final_direct['Target_Description'] = 'Direct'
hpv_positive_final_indirect['Target_Description'] = 'Indirect'
hpv_positive_final_results = pd.concat([hpv_positive_final_direct, hpv_positive_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_positive_final_results = hpv_positive_final_results.fillna('NA')
hpv_positive_final_results = hpv_positive_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

/var/folders/5p/swntgnbj3fbfxkx02kt3fq980000gn/T/ipykernel_46398/4020300996.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpv_positive_final_direct['Target_Description'] = 'Direct'


In [121]:
hpv_positive_final_results.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending = [ True, False ]).head(50)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),NUM_DIRECT_TARGETS_HIT,Number of risk or immediate neighbor genes targeted,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,Target_Description
106,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,NA,6.0,NA,100.000000,UNKNOWN,ATP binding,1.169028e-05,0.002129,Indirect
16,arsenic trioxide,"IKBKB, JUN, HDAC1, CDKN1A, MAPK1, CCND1, PML, ...","CYLD, EP300",NA,8.0,NA,80.000000,"inducer, inducer, antagonist","ATP binding, cAMP response element binding, co...",1.709173e-06,0.002129,Indirect
104,xl765,"PIK3CD, PIK3CG, PIK3CB, MTOR",PIK3CA,NA,4.0,NA,80.000000,UNKNOWN,"1-phosphatidylinositol-3-kinase activity, ATP ...",1.534269e-04,0.002129,Indirect
23,bosutinib,"HCK, FGR, MAP2K1, ABL1, LYN, SRC","PIK3CA, EP300",NA,6.0,NA,54.545455,"inhibitor, inhibitor","ATP binding, actin filament binding, ATP binding",1.552648e-03,0.002129,Indirect
24,brigatinib,"IGF1R, MET, ALK, ERBB4, INSR, EGFR, ABL1, ERBB...","PIK3CA,VEGFD, PIK3CA",NA,9.0,NA,100.000000,"inhibitor, binding, inhibitor, binding","ATP binding, amyloid-beta binding, actin filam...",1.590798e-05,0.002532,Indirect
37,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,10.0,NA,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",3.033567e-06,0.002532,Indirect
53,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,10.0,NA,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",3.033567e-06,0.002532,Indirect
55,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA,ANOS1,VEGFD, PIK3CA",NA,8.0,NA,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",8.743881e-05,0.002532,Indirect
61,nintedanib,"FGFR1, FGFR3, SRC, KDR, PDGFRB, LCK, FLT1, PDG...","PIK3CA,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",NA,12.0,NA,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, ATP binding, ATP binding",2.018575e-07,0.002532,Indirect
67,pd-166326,"FGFR1, SRC, PDGFRB, LCK, EGFR, KIT, ABL1, PDGF...","PIK3CA,GNB4,ANOS1,SH3KBP1,VEGFD, PIK3CA, EP300",NA,9.0,NA,100.000000,"inhibitor, inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",1.590798e-05,0.002532,Indirect


In [122]:
len(set(hpv_positive_final_results['DRUG']))

109

## HPV-

#### Genes

In [123]:
### combine hpv negative somatic genes and cnv genes
hpv_negative_som_genes['MUT_TYPE'] = 'SOMATIC'
hpv_negative_som_genes['gene_name'] = hpv_negative_som_genes['Gene']
hpv_negative_som_genes['q_value']= hpv_negative_som_genes['Adjusted_P_Value']
hpv_negative_som_genes['empirical_q_value'] = hpv_negative_som_genes['Adjusted_Empirical_P_Value']
hpv_negative_combined_genes = pd.concat([hpv_negative_genes, hpv_negative_som_genes], axis=0)
### aggregate by GENE to get unique genes with both mutation types
hpv_negative_combined_genes = hpv_negative_combined_genes.groupby('gene_name').agg({
    'MUT_TYPE': lambda x: ', '.join(x),
    'q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str),
    'empirical_q_value': lambda x: ', '.join(x.astype(str)) if len(x) > 1 else x.iloc[0].astype(str)
}).reset_index()
# hpv_negative_combined_genes.sort_values(by='q_value')

In [124]:
hpv_negative_combined_genes

,gene_name,MUT_TYPE,q_value,empirical_q_value
0,ACTL6A,AMPLIFICATION,4.6046768654042024e-207,0.0043585777302941
1,ADCY2,SOMATIC,0.0003377698987821,0.005960073448722
2,ADCY8,SOMATIC,0.0021845458816456,0.005960073448722
3,ADGRB3,SOMATIC,8.330155899454921e-05,0.005960073448722
4,ADIPOQ,AMPLIFICATION,8.446185897603275e-199,0.0043585777302941
...,...,...,...,...
218,ZNF676,SOMATIC,0.0239614999999452,0.005960073448722
219,ZNF804A,SOMATIC,0.003536214310732,0.005960073448722
220,ZNF804B,SOMATIC,5.057316865018672e-06,0.005960073448722
221,ZNF835,SOMATIC,1.0196847272849249e-05,0.005960073448722


In [125]:
### merge genes with number of articles, pubmed id from literature data
hpv_negative_gene_results_with_lit = pd.merge(hpv_negative_combined_genes, extracted_target_df_combined, how = 'left', left_on='gene_name', right_on='GENE')
hpv_negative_gene_results_with_lit.drop(columns = ['INDEX'], inplace = True)

In [126]:
hpv_negative_gene_results_with_lit = hpv_negative_gene_results_with_lit[hpv_negative_gene_results_with_lit['NUMBER_OF_ARTICLES']>0]

In [127]:
hpv_negative_gene_results_with_lit.sort_values(by = ['empirical_q_value', 'q_value'], ascending=[True, True], inplace=True)

In [128]:
hpv_negative_gene_results_with_lit.drop(columns=['INDEX', 'GENE'], errors='ignore', inplace=True)

In [129]:
hpv_negative_gene_results_with_lit.to_csv('Results/HPV negative gene results.csv')

In [130]:
hpv_negative_gene_results_with_lit

,gene_name,MUT_TYPE,q_value,empirical_q_value,PMID,NUMBER_OF_ARTICLES
50,EIF4G1,AMPLIFICATION,1.4051416817717394e-199,0.0043585777302941,14676830,1.0
192,SOX2,AMPLIFICATION,1.960361075687212e-205,0.0043585777302941,15942670,1.0
44,DVL3,AMPLIFICATION,2.0584024908359247e-200,0.0043585777302941,"14676830, 16865240",2.0
170,PRKCI,AMPLIFICATION,3.026251441733312e-201,0.0043585777302941,17990328,1.0
158,PDCD10,AMPLIFICATION,3.0744600814299213e-196,0.0043585777302941,17409414,1.0
25,CLDN1,AMPLIFICATION,4.2419726156021995e-197,0.0043585777302941,"15170668, 17091452",2.0
11,BCL6,AMPLIFICATION,5.884151323631562e-198,0.0043585777302941,"11224600, 11420458, 14685876, 17429099",4.0
176,RFC4,AMPLIFICATION,8.446185897603275e-199,0.0043585777302941,16467079,1.0
163,PIK3CA,"AMPLIFICATION, SOMATIC","1.5505691778987169e-208, 8.841194236005348e-40","0.0043585777302941, 0.005960073448722","11358835, 11836556, 11959846, 14581353, 155436...",17.0
204,TP53,SOMATIC,0.0,0.005960073448722,"11390535, 11445847, 11445859, 11916556, 125898...",29.0


In [131]:
len(set(hpv_negative_gene_results_with_lit['gene_name']))

28

In [132]:
### export top genes to ouput final tables
hpv_negative_gene_results_with_lit.to_csv('Results/Final Results/HPV Negative validated genes.csv')

#### Direct

In [133]:
### combine all hpv negative direct drug candidates
hpv_negative_final_direct = pd.concat([hpv_negative_direct_drug_candidates, hpv_negative_som_direct_drug_candidates])
### group by drug and comma seperate genes and mutation type
### columns: DRUG	GENE_TARGET	NUM_DIRECT_TARGETS_HIT	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	GENE_GISTIC	GENE_normalized_gistic_score	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_p_value      

hpv_negative_final_direct = hpv_negative_final_direct.groupby('DRUG').agg({'GENE_TARGET': lambda x: ', '.join(x),
                                               'MUT_TYPE': lambda x: ', '.join(x.unique()), ### unique mutation types per drug
                                                  'NUM_DIRECT_TARGETS_HIT': 'first',
                                                    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
                                                    'PERCENTAGE_OF_TARGETS_HIT': 'first',
                                                    'ACTION': 'first',
                                                    'SPECIFIC_FUNCTION': 'first',
                                                    'drug_hypergeom_p_value': 'first',
                                                    'drug_hypergeom_fdr': 'first',
                                                    'drug_empirical_p_value': 'first',
                                                    'drug_empirical_fdr': 'first',
                                                    }).reset_index()

In [134]:
### validate hpv negative direct drug candidates with literature data
hpv_negative_final_direct['PMID'] = ''
hpv_negative_final_direct['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_direct['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_direct.iterrows():
    gene_targets = row['GENE_TARGET'].split(', ')
    gene_targets = list(set(gene_targets))
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(', ')
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_direct.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_direct.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_direct.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles

    

In [135]:
hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0].sort_values(by='NUMBER_OF_ARTICLES', ascending=False)

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,2.544891e-06,1.135264e-03,0.00002,0.006691,"12708486, 16969480, 11445847, 15810068, 168478...",29,TP53
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.442356e-04,2.598460e-02,0.00013,0.023879,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
18,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00045,0.048384,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
19,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00034,0.045186,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
10,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA5, EPHA2, TGFBR2, EPHA7","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,5.967681e-10,7.986463e-07,0.00001,0.002755,"17990328, 18030354, 12494475, 18425361, 184857...",6,"EPHA2, PRKCI"
6,Dasatinib,"EPHA5, EPHA2",SOMATIC,2.0,23.0,8.695652,inhibitor,ATP binding,5.499430e-08,5.724296e-05,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
15,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.605554e-10,4.499242e-06,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.004451e-07,6.273132e-05,0.00001,0.002755,17990328,1,PRKCI
3,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.350142e-05,6.796138e-03,0.00003,0.009066,16857411,1,CASP8
11,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.820700e-05,1.558815e-02,0.00013,0.023879,16135921,1,SELP


In [136]:
### ensure that only drugs with NUMBER_OF_ARTICLES > 0 are saved, so that they have literature support
hpv_negative_final_direct= hpv_negative_final_direct[hpv_negative_final_direct['NUMBER_OF_ARTICLES'] > 0]
### save results
hpv_negative_final_direct.to_csv('Results/HPV Negative direct results.csv')

In [137]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,2.544891e-06,1.135264e-03,0.00002,0.006691,"12708486, 16969480, 11445847, 15810068, 168478...",29,TP53
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.004451e-07,6.273132e-05,0.00001,0.002755,17990328,1,PRKCI
3,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.350142e-05,6.796138e-03,0.00003,0.009066,16857411,1,CASP8
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.442356e-04,2.598460e-02,0.00013,0.023879,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
6,Dasatinib,"EPHA5, EPHA2",SOMATIC,2.0,23.0,8.695652,inhibitor,ATP binding,5.499430e-08,5.724296e-05,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
10,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA5, EPHA2, TGFBR2, EPHA7","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,5.967681e-10,7.986463e-07,0.00001,0.002755,"17990328, 18030354, 12494475, 18425361, 184857...",6,"EPHA2, PRKCI"
11,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.820700e-05,1.558815e-02,0.00013,0.023879,16135921,1,SELP
15,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.605554e-10,4.499242e-06,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
18,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00045,0.048384,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
19,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00034,0.045186,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


#### Indirect

In [138]:
### combine all hpv negative indirect drug candidates
hpv_negative_final_indirect = pd.concat([hpv_negative_indirect_drug_candidates, hpv_negative_som_indirect_drug_candidates])
hpv_negative_final_indirect['ACTION'] = hpv_negative_final_indirect['ACTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['SPECIFIC_FUNCTION'] = hpv_negative_final_indirect['SPECIFIC_FUNCTION'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_hypergeom_fdr'] = hpv_negative_final_indirect['drug_hypergeom_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['drug_empirical_fdr'] = hpv_negative_final_indirect['drug_empirical_fdr'].fillna('UNKNOWN')
hpv_negative_final_indirect['MUT_TYPE'] = hpv_negative_final_indirect['MUT_TYPE'].fillna('UNKNOWN')

### group by drug and comma seperate genes and mutation type
### columns: DRUG	CONNECTED_TO (risk gene)
# Number of risk or immediate neighbor genes
# targeted	total_genes_targeted_in_drugbank  
# PERCENTAGE_OF_TARGETS_HIT
# Number of indirect genes connected to this risk gene  
# GENE_TARGET	
# GENE_Cohort_Frequency   
# GENE_Normalized_Count	
# GENE_Normalized_Cohort_Frequency

hpv_negative_final_indirect = hpv_negative_final_indirect.groupby (['DRUG']).agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'Number of risk or immediate neighbor genes targeted': 'first',
    'total_genes_targeted_in_drugbank': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'first',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'MUT_TYPE': lambda x: ', '.join(x)
}).reset_index()

In [139]:
'artenimol' in hpv_negative_final_indirect['DRUG']

False

In [140]:
hpv_negative_final_indirect.sort_values(by = 'drug_empirical_fdr', ascending = True).head(20)

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE
61,nadh,"NDUFA8, NDUFS5, NDUFS2, NDUFA2, NDUFB10, NDUFV...",NDUFB5,58,144,40.277778,binder,"NADH dehydrogenase (ubiquinone) activity, 4 ir...",2.056983e-05,0.002755,AMPLIFICATION
13,biotin,"ACACA, HLCS, PC,ACACB, MCCC1, PCCB, PCCA, MCCC2","MCCC1,EHHADH",8,8,100.000000,"cofactor, substrate","acetyl-CoA carboxylase activity, ATP binding",1.011251e-03,0.002755,AMPLIFICATION
25,cholic acid,"CES1,PLA2G1B","NCEH1,PLD1",16,22,72.727273,inhibitor,carboxylesterase activity,1.055693e-04,0.002755,AMPLIFICATION
32,erdafitinib,"FGFR1, FGFR4, FGFR3, KDR, PDGFRB, KIT, CSF1R, ...","PIK3CA, PIK3CA",10,10,100.000000,"inhibitor, substrate, inhibitor, substrate","ATP binding, ATP binding",8.944839e-05,0.004073,"AMPLIFICATION, SOMATIC"
68,pazopanib,"FGFR3, KDR, PDGFRB, ITK, FLT1, KIT, PDGFRA, FL...","PIK3CA,THPO, PIK3CA, EPHA2",9,10,90.000000,"inhibitor, inhibitor, inhibitor","ATP binding, ATP binding, fibroblast growth fa...",2.171896e-03,0.004073,"AMPLIFICATION, SOMATIC, SOMATIC"
69,pd-166326,"FGFR1, SRC, PDGFRB, LCK, EGFR, KIT, ABL1, PDGF...","PIK3CA,GNB4,CDKN2A, TP53, PIK3CA, KRT5",9,9,100.000000,"inhibitor, inhibitor, inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",3.601417e-04,0.004073,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC, SOMATIC"
74,ponatinib,"FGFR1, FGFR4, FGFR3, SRC, KDR, LCK, KIT, RET, ...","PIK3CA,THPO,CDKN2A, PIK3CA, RHOA",14,15,93.333333,"inhibitor, inhibitor, inhibitor","ATP binding, actin filament binding, ATP bindi...",8.707242e-06,0.004073,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC"
52,lucitanib,"FGFR1, FGFR3, KDR, PDGFRB, FLT1, PDGFRA, FGFR2...","PIK3CA, PIK3CA",8,8,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",1.359296e-03,0.004073,"AMPLIFICATION, SOMATIC"
50,lenvatinib,"FGFR1, FGFR4, FGFR3, KDR, FLT1, KIT, RET, PDGF...","PIK3CA, PIK3CA",10,10,100.000000,"inhibitor, inhibitor","ATP binding, ATP binding",8.944839e-05,0.004073,"AMPLIFICATION, SOMATIC"
76,pralsetinib,"FGFR1, JAK2, KDR, PDGFRB, RET, JAK1, FGFR2, NT...","PIK3CA, PIK3CA",10,11,90.909091,"inhibitor, inhibitor","ATP binding, acetylcholine receptor binding, A...",5.608025e-04,0.004073,"AMPLIFICATION, SOMATIC"


In [141]:
### extract the unique number of GENE_TARGETs from , seperated list in each of the GENE_TARGET column
def extract_unique_gene_targets(df, column_name):
    return len(set([gene.strip() for sublist in df[column_name].dropna().str.split(',') for gene in sublist]))

hpv_negative_indirect_genes = extract_unique_gene_targets(hpv_negative_final_indirect, 'GENE_TARGET')
print(hpv_negative_indirect_genes)

432


In [142]:
### add in literature validation columns
hpv_negative_final_indirect['PMID'] = ''
hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    gene_targets = row['GENE_TARGET'].split(',')
    gene_targets = [gene.strip() for gene in gene_targets]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in gene_targets:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'NUMBER_OF_ARTICLES'] = number_of_articles


### validate risk genes
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect['RISK_GENE_PMID'] = ''
hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] = 0
hpv_negative_final_indirect['RISK_GENE_LITERATURE_GENE_TARGETS'] = ''
for index, row in hpv_negative_final_indirect.iterrows():
    risk_genes = row['CONNECTED_TO (risk gene)'].split(',')
    risk_genes = [gene.strip() for gene in risk_genes]
    pmids_set = set()
    literature_gene_targets = set()
    number_of_articles = 0
    for gene in risk_genes:
        matched_rows = extracted_target_df_combined[extracted_target_df_combined['GENE'] == gene]
        for _, matched_row in matched_rows.iterrows():
            pmids = matched_row['PMID'].split(',')
            pmids = [pmid.strip() for pmid in pmids]
            pmids_set.update(pmids)
            number_of_articles += matched_row['NUMBER_OF_ARTICLES']
            literature_gene_targets.add(matched_row['GENE'])
    
    hpv_negative_final_indirect.at[index, 'RISK_GENE_LITERATURE_GENE_TARGETS'] = ', '.join(literature_gene_targets)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_PMID'] = ', '.join(pmids_set)
    hpv_negative_final_indirect.at[index, 'RISK_GENE_NUMBER_OF_ARTICLES'] = number_of_articles

### make sure only drugs with literature support for both drug targets and risk genes are saved
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['NUMBER_OF_ARTICLES'] > 0]
hpv_negative_final_indirect = hpv_negative_final_indirect[hpv_negative_final_indirect['RISK_GENE_NUMBER_OF_ARTICLES'] > 0]

hpv_negative_final_indirect.to_csv('Results/HPV Negative indirect results.csv', index=False)



In [143]:
hpv_negative_final_indirect.sort_values(by = ['drug_empirical_fdr', 'PERCENTAGE_OF_TARGETS_HIT'], ascending=[True, False]).head(50)[['DRUG', 'drug_empirical_fdr','LITERATURE_GENE_TARGETS', 'RISK_GENE_PMID','RISK_GENE_LITERATURE_GENE_TARGETS','RISK_GENE_NUMBER_OF_ARTICLES','MUT_TYPE']]

,DRUG,drug_empirical_fdr,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_LITERATURE_GENE_TARGETS,RISK_GENE_NUMBER_OF_ARTICLES,MUT_TYPE
16,brigatinib,0.004073,"EGFR, ERBB2, MET, IGF1R, ALK, ERBB4","18376308, 16380997, 15083191, 11836556, 117204...","PIK3CA, CDKN2A",50,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC"
32,erdafitinib,0.004073,"KIT, FGFR1, FGFR3, RET, FGFR2, FGFR4, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
50,lenvatinib,0.004073,"KIT, FGFR1, FGFR3, RET, FGFR2, FGFR4, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
52,lucitanib,0.004073,"FGFR1, FGFR2, FGFR3, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
64,nintedanib,0.004073,"SRC, FGFR1, LYN, FGFR3, FGFR2, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
69,pd-166326,0.004073,"KIT, EGFR, SRC, FGFR1, PDGFRA","18376308, 12708486, 16380997, 15083191, 169694...","PIK3CA, TP53, CDKN2A",79,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC, SOMATIC"
92,sorafenib,0.004073,"KIT, EGFR, FGFR1, BRAF, RET","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC, SOMATIC"
74,ponatinib,0.004073,"KIT, SRC, FGFR1, LYN, FGFR3, RET, FGFR2, FGFR4...","18376308, 12082550, 16380997, 15083191, 118365...","PIK3CA, RHOA, CDKN2A",53,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC"
76,pralsetinib,0.004073,"FGFR1, JAK2, RET, FGFR2, NTRK3, NTRK1","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC"
101,tivozanib,0.004073,"KIT, FGFR1, MET, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",PIK3CA,34,"AMPLIFICATION, SOMATIC, SOMATIC"


In [144]:
hpv_negative_final_indirect[hpv_negative_final_indirect['DRUG'] == 'XL765']

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS


#### overall

In [145]:
hpv_negative_final_direct[hpv_negative_final_direct['GENE_TARGET'] == 'PIK3CA']

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,0.000144,0.025985,0.00013,0.023879,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
18,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,0.000404,0.042006,0.00045,0.048384,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
19,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,0.000404,0.042006,0.00034,0.045186,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


In [146]:
hpv_negative_final_direct

,DRUG,GENE_TARGET,MUT_TYPE,NUM_DIRECT_TARGETS_HIT,TOTAL_TARGETS_IN_DRUGBANK,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_p_value,drug_hypergeom_fdr,drug_empirical_p_value,drug_empirical_fdr,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS
0,Acetylsalicylic acid,TP53,SOMATIC,1.0,20.0,5.000000,inducer,14-3-3 protein binding,2.544891e-06,1.135264e-03,0.00002,0.006691,"12708486, 16969480, 11445847, 15810068, 168478...",29,TP53
2,Bisindolylmaleimide I,PRKCI,AMPLIFICATION,1.0,19.0,5.263158,None,ATP binding,1.004451e-07,6.273132e-05,0.00001,0.002755,17990328,1,PRKCI
3,Bryostatin 1,CASP8,SOMATIC,1.0,9.0,11.111111,inhibitor,cysteine-type endopeptidase activity,2.350142e-05,6.796138e-03,0.00003,0.009066,16857411,1,CASP8
4,Caffeine,PIK3CA,SOMATIC,1.0,15.0,6.666667,inhibitor,1-phosphatidylinositol-3-kinase activity,1.442356e-04,2.598460e-02,0.00013,0.023879,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
6,Dasatinib,"EPHA5, EPHA2",SOMATIC,2.0,23.0,8.695652,inhibitor,ATP binding,5.499430e-08,5.724296e-05,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
10,Fostamatinib,"MAP3K13, PRKCI, TNIK, EPHA5, EPHA2, TGFBR2, EPHA7","AMPLIFICATION, SOMATIC",3.0,300.0,1.000000,inhibitor,ATP binding,5.967681e-10,7.986463e-07,0.00001,0.002755,"17990328, 18030354, 12494475, 18425361, 184857...",6,"EPHA2, PRKCI"
11,Heparin,SELP,SOMATIC,1.0,12.0,8.333333,inhibitor,calcium ion binding,7.820700e-05,1.558815e-02,0.00013,0.023879,16135921,1,SELP
15,Regorafenib,EPHA2,SOMATIC,1.0,18.0,5.555556,inhibitor,ATP binding,9.605554e-10,4.499242e-06,0.00001,0.004073,"18030354, 12494475, 18425361, 18485799, 16309192",5,EPHA2
18,Wortmannin,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00045,0.048384,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
19,XL765,PIK3CA,AMPLIFICATION,1.0,5.0,20.000000,None,1-phosphatidylinositol-3-kinase activity,4.035547e-04,4.200556e-02,0.00034,0.045186,"15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


In [147]:
hpv_negative_final_indirect

,DRUG,GENE_TARGET,CONNECTED_TO (risk gene),Number of risk or immediate neighbor genes targeted,total_genes_targeted_in_drugbank,PERCENTAGE_OF_TARGETS_HIT,ACTION,SPECIFIC_FUNCTION,drug_hypergeom_fdr,drug_empirical_fdr,MUT_TYPE,PMID,NUMBER_OF_ARTICLES,LITERATURE_GENE_TARGETS,RISK_GENE_PMID,RISK_GENE_NUMBER_OF_ARTICLES,RISK_GENE_LITERATURE_GENE_TARGETS
2,acetylsalicylic acid,"PRKAA1, PCNA, CASP3, CASP1, IKBKB, NFKBIA, PTG...","TP53, HRAS, PIK3CA, KRT5, UGT2B4",14,19,73.684211,"activator, downregulator, inhibitor, inhibitor...",[hydroxymethylglutaryl-CoA reductase (NADPH)] ...,0.001135,0.006691,"SOMATIC, SOMATIC, SOMATIC, SOMATIC, SOMATIC","12854173, 15566678, 11854069, 15039910, 175424...",113,"NFKBIA, PTGS2, AKR1C1, MYC, TP53, CCND1, PCNA","18376308, 12708486, 16380997, 16969480, 114458...",47,"PIK3CA, TP53, HRAS"
4,ag-24322,"CDK1,CDK2, CDK4","CDKN2A,CDKN2B",3,3,100.000000,inhibitor,ATP binding,0.010932,0.022483,DELETION,"16496417, 17487385, 15645429, 16452236, 118540...",29,"CDK4, CDK1, CDK2","14586645, 17117177, 11309301, 17079134, 176739...",18,"CDKN2A, CDKN2B"
5,alsterpaullone,"CDK1, CDK5,CDK2","CDKN2A,CDKN2B",3,4,75.000000,inhibitor,"ATP binding, acetylcholine receptor activator ...",0.034197,0.022483,DELETION,"11453659, 16496417, 15645429, 11585773, 164522...",16,"CDK1, CDK2, CDK5","14586645, 17117177, 11309301, 17079134, 176739...",18,"CDKN2A, CDKN2B"
6,altiratinib,"MET, KDR, NTRK1, NTRK3,TEK","PIK3CA,THPO",5,5,100.000000,"inhibitor, antagonist",ATP binding,0.042006,0.045186,AMPLIFICATION,"15735049, 11752453, 18349821, 16483615, 112796...",15,"MET, NTRK1, NTRK3","15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA
7,alvocidib,"CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5, CDK8...","CDKN2B, TP53, PIK3CA, KRT5",6,12,50.000000,"inhibitor, inhibitor, UNKNOWN, UNKNOWN","ATP binding, ATP binding, 7SK snRNA binding, a...",0.015588,0.016394,"DELETION, SOMATIC, SOMATIC, SOMATIC","11854069, 17208308, 17635577, 12833132, 183072...",366,"CDK6, EGFR, CDK4, CDK2, CDK5, CDK1","18376308, 12708486, 16380997, 16969480, 114458...",48,"PIK3CA, TP53, CDKN2B"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,tetrabromo-2-benzotriazole,"MAP2K1, LCK, RPS6KB1, AKT1,PRKCA, MAPK11, MAPK...","PIK3CA,GNB4,DVL3,RFC4,MECOM,CDKN2A, TP53, PIK3...",13,18,72.222222,"inhibitor, inhibitor, inhibitor, inhibitor, in...","ATP binding, 14-3-3 protein binding, [hydroxym...",0.001126,0.004804,"AMPLIFICATION,DELETION, SOMATIC, SOMATIC, SOMA...","12532415, 15896313, 16002527, 17974918, 145813...",24,"CHEK1, MAPK8, AKT1","18376308, 12708486, 12082550, 16380997, 150831...",85,"PIK3CA, RHOA, DVL3, CDKN2A, RFC4, TP53"
101,tivozanib,"FGFR1, MET, KDR, PDGFRB, FLT1, KIT, PDGFRA, FL...","PIK3CA, PIK3CA, KHDRBS2",10,11,90.909091,"inhibitor, inhibitor, UNKNOWN","ATP binding, ATP binding, ATP binding",0.000561,0.004073,"AMPLIFICATION, SOMATIC, SOMATIC","18349821, 17075124, 17545628, 16342249, 117058...",50,"KIT, FGFR1, MET, PDGFRA","15700036, 16676365, 16807070, 11358835, 175493...",34,PIK3CA
104,trilaciclib,"CDK5,CDK2, CDK4, CDK6, CDK7, CDK6, CDK9, CDK5,...","CDKN2A,CDKN2B, TP53, PIK3CA",4,6,66.666667,"inhibitor, inhibitor, inhibitor","acetylcholine receptor activator activity, ATP...",0.015588,0.019607,"DELETION, SOMATIC, SOMATIC","16496417, 17487385, 16452236, 11854069, 120173...",68,"CDK6, CDK4, CDK2, CDK5","18376308, 12708486, 15083191, 16380997, 169694...",64,"CDKN2A, TP53, PIK3CA, CDKN2B"
108,xl999,"FGFR1, FGFR3, KDR, PDGFRB, RET, FLT3",PIK3CA,6,6,100.000000,UNKNOWN,ATP binding,0.015588,0.017844,SOMATIC,"16209369, 14962731, 16350727, 10999773, 119531...",55,"RET, FGFR1, FGFR3","15700036, 16676365, 16807070, 11358835, 175493...",17,PIK3CA


In [148]:
### combined direct and indirect final results
### final columns: DRUG   GENE_TARGET CONNECTED_TO (risk gene)	NUM_DIRECT_TARGETS_HIT  
# Number of risk or immediate neighbor genes targeted	TOTAL_TARGETS_IN_DRUGBANK	PERCENTAGE_OF_TARGETS_HIT	
# ACTION	SPECIFIC_FUNCTION	drug_hypergeom_fdr	drug_empirical_fdr	
### column for Target description: direct, or indirect
hpv_negative_final_direct['Target_Description'] = 'Direct'
hpv_negative_final_indirect['Target_Description'] = 'Indirect'
hpv_negative_final_results = pd.concat([hpv_negative_final_direct, hpv_negative_final_indirect], ignore_index=True)
### replace any null with 'NA' in the hwole dataframe
hpv_negative_final_results = hpv_negative_final_results.fillna('NA')
hpv_negative_final_results = hpv_negative_final_results.groupby('DRUG').agg({
    'GENE_TARGET': lambda x: ', '.join(x),
    'CONNECTED_TO (risk gene)': lambda x: ', '.join(x),
    'NUM_DIRECT_TARGETS_HIT': 'first',
    'Number of risk or immediate neighbor genes targeted': 'first',
    'TOTAL_TARGETS_IN_DRUGBANK': 'first',
    'PERCENTAGE_OF_TARGETS_HIT': 'max',
    'ACTION': lambda x: ', '.join(x),
    'SPECIFIC_FUNCTION': lambda x: ', '.join(x),
    'drug_hypergeom_fdr': 'max',
    'drug_empirical_fdr': 'max',
    'Target_Description': lambda x: ', '.join(x)
}).reset_index()

In [149]:
len(set(hpv_negative_final_results['DRUG']))

77